
#1. Exploracion de los datos Wanderbricks

Es necesario revisar las tablas del dataset de Wanderbricks para lograr entender la cantidad y el tipo de informacion que se manejan a continuacion: 

* **users:** 124,509 usuarios en la plataforma.
* **hosts:** 19,384 anfitriones de propiedades.
* **properties:** 18,163 propiedades publicadas.
* **bookings:** 72,247 reservas registradas.
* **payments:** 49,638 pagos procesados.
* **reviews:** 99,793 reseñas de clientes.
* **clickstream:** 100,000 eventos de clics de los usuarios

### Tipos de datos que encontraremos

* **Tablas estructuradas:** `users`, `properties`, `bookings` y `payments` siendo conectadas entre si con ID.

* **Datos anidados:** en el `clickstream` la columna `metadata` viene con el dispositivo y la pagina original siendo agrupados.

* **Texto libre:** En `reviews` se tienen las opiniones escritas de los clientes y sus respectivas puntuaciones. 



#2. ¿Porque elegi un Lakehouse?
Se evaluo las diferentes opciones para poder hacer un guardado de los datos en wanderbricks:
    - **Base de datos relacional:** Funciona bien para conectar reservas con pagos, pero es muy costosa para manejar los 100,000 datos de clics e informacion anidada. 

    - **Base de datos NoSQL:** Es perfecta para poder guardar los clics y comentarios en texto libre, sin embargo es mala para hacer cruces rapidos de dinero y reportes.

    - **Nuestra eleccion - Lakehouse:** Es la mejor opcion a la hora de realizar estos procesos porque permite guardar todo junto sin perder velocidad ni capacidad para hacer consultas SQL.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS mi_lakehouse")
print("Base de datos lista :)")

Base de datos lista :)


In [0]:

spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_users AS SELECT * FROM samples.wanderbricks.users")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_hosts AS SELECT * FROM samples.wanderbricks.hosts")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_properties AS SELECT * FROM samples.wanderbricks.properties")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_bookings AS SELECT * FROM samples.wanderbricks.bookings")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_payments AS SELECT * FROM samples.wanderbricks.payments")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_reviews AS SELECT * FROM samples.wanderbricks.reviews")
spark.sql("CREATE TABLE IF NOT EXISTS mi_lakehouse.bronce_clickstream AS SELECT * FROM samples.wanderbricks.clickstream")

print(" las tablas ya estan copiadas en la capa bronce.")

 las tablas ya estan copiadas en la capa bronce.


#Capa plata y desanidacion de los datos 

En esta etapa tomamos la tabla `bronce_clickstream` y luego se extrajo la informacion que viene guardada dentro del objeto `metadata`.
Se separo un campo en dos columnas independientes `device` y `referrer`, para poder consultarlas directamente sin ningun problema.

In [0]:
#Se desanida la columna de metadata y luego se guarda en la capa plata
df_clickstream = spark.table("mi_lakehouse.bronce_clickstream")

df_plata_clickstream = df_clickstream.select(
    "user_id",
    "property_id",
    "metadata.device",
    "metadata.referrer"
)

df_plata_clickstream.write.format("delta").mode("overwrite").saveAsTable("mi_lakehouse.plata_clickstream")

print("La capa plata esta lista, la tabla fue desanidada de manera exitosa.")

La capa plata esta lista, la tabla fue desanidada de manera exitosa.


In [0]:
display(spark.sql("SELECT * FROM mi_lakehouse.plata_clickstream LIMIT 5"))

user_id,property_id,device,referrer
16390,5135,desktop,email
3655,7805,desktop,direct
100094,17349,mobile,direct
102187,6128,tablet,ad
40863,3438,mobile,google



#Capa Oro: Modelo unificado de Datos
para poder facilitar el uso que les de los respectivos equipos de analisis, se hizo una union en la clave en una sola tabla consolidada.
Uni los datos de las reservas con las propiedades, los pagos que hayan realizado, opiniones de usuarios e informacion, haciendo que resolver dudas en cuanto al negocio de una manera mas organizada y rapida en SQL.


In [0]:
oro_df = spark.sql("""
    SELECT 
        b.booking_id,
        b.user_id,
        b.property_id,
        b.status AS estado_reserva,
        u.email AS email_usuario
    FROM mi_lakehouse.bronce_bookings b
    LEFT JOIN mi_lakehouse.bronce_users u ON b.user_id = u.user_id
""")

oro_df.write.format("delta").mode("overwrite").saveAsTable("mi_lakehouse.oro_consolidado")

print("La capa oro fue creada de manera exitosa :)")

La capa oro fue creada de manera exitosa :)


In [0]:
oro_df = spark.sql("""
    SELECT 
        b.booking_id,
        b.user_id,
        b.property_id,
        b.status AS estado_reserva,
        u.email AS email_usuario,
        u.country AS pais_usuario,
        p.destination_id,
        pay.status AS estado_pago,
        r.rating,
        r.comment AS resena_texto
    FROM mi_lakehouse.bronce_bookings b
    LEFT JOIN mi_lakehouse.bronce_users u ON b.user_id = u.user_id
    LEFT JOIN mi_lakehouse.bronce_properties p ON b.property_id = p.property_id
    LEFT JOIN mi_lakehouse.bronce_payments pay ON b.booking_id = pay.booking_id
    LEFT JOIN mi_lakehouse.bronce_reviews r ON b.booking_id = r.booking_id
""") 

oro_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mi_lakehouse.oro_consolidado")

print(" La capa oro fue creada de manera exitosa :) ")


 La capa oro fue creada de manera exitosa :) 


# Evidencia de propiedades del formato Delta Lake
1. **Atomicidad:** Logra garantizar que las modificaciones en las tablas se ejecuten por completo y/o se cancelen sin dejar datos corruptos 

2. **Versionamiento:** Registro auditor mediante `Describe History` el cual nos permite consultar estados anteriores de cada dato

3. **Schema Evolution:** Capacidad de modificar o editar las columnas de las tablas en la opcion `mergeSchema`.

In [0]:
from pyspark.sql.functions import lit

# 1. Crear la tabla de trabajo en el esquema default
oro_df = spark.sql("""
    SELECT 
        b.booking_id,
        b.user_id,
        b.property_id,
        b.status AS estado_reserva,
        u.email AS email_usuario,
        u.country AS pais_usuario,
        p.destination_id,
        pay.status AS estado_pago,
        r.rating,
        r.comment AS resena_texto
    FROM mi_lakehouse.bronce_bookings b
    LEFT JOIN mi_lakehouse.bronce_users u ON b.user_id = u.user_id
    LEFT JOIN mi_lakehouse.bronce_properties p ON b.property_id = p.property_id
    LEFT JOIN mi_lakehouse.bronce_payments pay ON b.booking_id = pay.booking_id
    LEFT JOIN mi_lakehouse.bronce_reviews r ON b.booking_id = r.booking_id
""")

oro_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("oro_consolidado")

print("--- Capa Oro lista para pruebas ---")

# 2. Atomicidad: Inserción
df_insert = spark.table("oro_consolidado").limit(1) \
    .withColumn("booking_id", lit("BOOKING_PRUEBA_999"))

df_insert.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("oro_consolidado")

print("1. Inserción atómica completada.")

# 3. Evolución de Esquema
df_nuevo_esquema = spark.table("oro_consolidado") \
    .limit(5) \
    .withColumn("categoria_descuento", lit("STANDARD"))

df_nuevo_esquema.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("oro_consolidado")

print("2. Evolución de esquema aplicada con exito.")

# 4. Historial (Time Travel)
display(spark.sql("DESCRIBE HISTORY oro_consolidado"))

--- Capa Oro lista para pruebas ---


---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4567720519763650>, line 38
     31 # 2. Atomicidad: Inserción
     32 df_insert = spark.table("oro_consolidado").limit(1) \
     33     .withColumn("booking_id", lit("BOOKING_PRUEBA_999"))
     35 df_insert.write \
     36     .format("delta") \
     37     .mode("append") \
---> 38     .saveAsTable("oro_consolidado")
     40 print("1. Inserción atómica completada.")
     42 # 3. Evolución de Esquema

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callb

Consultas Analiticas del negocio (Capa Oro)
Ahora bien, se ejecutara 5 consultas analiticas clave sobre la tabla consolidada para dar respuesta a las preguntas de Wanderbricks

In [0]:
#1. Total de reservas por estado
print("Total de reservas por estado")
display(spark.sql("""
    SELECT estado_reserva, COUNT(*) AS total_reservas
    FROM oro_consolidado
    GROUP BY estado_reserva
    ORDER BY total_reservas DESC
"""))

#2. Distribucion de usuarios por cada pais
print("Distribucion de usuarios por cada pais")
display(spark.sql("""
    SELECT pais_usuario, COUNT(*) AS total_usuarios
    FROM oro_consolidado
    WHERE pais_usuario IS NOT NULL
    GROUP BY pais_usuario
    ORDER BY total_usuarios DESC 
    LIMIT 10 
    """))

#3. Promedio de calificacion x reserva
print("Promedio de calificacion x reserva")
display(spark.sql("""
    SELECT estado_reserva, ROUND(AVG(rating), 2) AS rating_promedio
    FROM oro_consolidado
    WHERE rating IS NOT NULL
    GROUP BY estado_reserva
    """))

#4. Estado de pagos
print("Estados de pago")
display(spark.sql("""
    SELECT estado_pago, COUNT(*) AS cantidad 
    FROM oro_consolidado
    WHERE estado_pago IS NOT NULL
    GROUP BY estado_pago
    """))

#5. Destinos con mas reservas
print("Destinos con mas reservas")
display(spark.sql("""
    SELECT destination_id, COUNT(booking_id) AS total_reservas
    FROM oro_consolidado
    WHERE destination_id IS NOT NULL
    GROUP BY destination_id
    ORDER BY total_reservas DESC
    LIMIT 5
    """))
    

Total de reservas por estado


estado_reserva,total_reservas
pending,53732
confirmed,31904
cancelled,26061
completed,12072


Distribucion de usuarios por cada pais


pais_usuario,total_usuarios
India,23002
China,22287
United States,5358
Indonesia,4362
Pakistan,3957
Nigeria,3735
Brazil,3585
Bangladesh,2728
Mexico,2177
Ethiopia,2072


Promedio de calificacion x reserva


estado_reserva,rating_promedio
pending,3.0
confirmed,3.02
completed,2.99
cancelled,3.01


Estados de pago


estado_pago,cantidad
completed,74229
refunded,6060
failed,817


Destinos con mas reservas


destination_id,total_reservas
3800,12372
2400,11327
3300,11040
3750,9685
1000,7636
